# Ingest results data for all measurement stations in Washington

Now we'll take the workflow we used in our single-site test, and expand it to all the sites in our dataset. I'm migrating the logic from `notebooks\01_raw_to_bronze_sites.ipynb` and `notebooks\01b_raw_to_bronze_sites_data_test .ipynb` into a src/hydroflow module. 

My goal is to have a set of logic that lets me put in the desired query parameters, and get data files in data/bronze. At this point, we're not worried about combining with previous data. We'll tackle that later. 

In [12]:
!pip install dataretrieval
!pip install duckdb
!pip install pandas

In [13]:
# This tells the container's Python to install the current directory in 'editable' mode
# This lets us access the src/hydroflow package from within the notebooks, and it also
# means that if we make changes to the code in src/hydroflow, we don't have to re-install
# the package to get those changes to be reflected in the notebooks.
!pip install -e /home/jovyan/work

Obtaining file:///home/jovyan/work
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for hydroflow (pyproject.toml) ... done
  Created wheel for hydroflow: filename=hydroflow-0.1.0-0.editable-py3-none-any.whl size=4869 sha256=2ed3c11ae45b348ca78a7be13a64d33f503c67ca3ff0fa167abdce51d8cd897f
  Stored in directory: /tmp/pip-ephem-wheel-cache-5tgln2_f/wheels/b3/ef/92/093562baef6fce9b92cfc59927bf9e82e306ed148e873389c0
Successfully built hydroflow
  Attempting uninstall: hydroflow
    Found existing installation: hydroflow 0.1.0
    Uninstalling hydroflow-0.1.0:
      Successfully uninstalled hydroflow-0.1.0


## How many queries?

When you're planning a big data harvest like this, its important to be a good data citizen. I can't repeatedly slam the API for multi-MB queries ten times a second. With >17k measurement sites in only the state of Washington, its important to break that query up.

We need to balance the number of queries with the size of the data we are requesting, and spread it over a suitable period of time. While I don't expect there to be throttling (or a temporary IP ban!) on the WQP's server, it is a possibility. When I've done scraping projects in the past, there are tricks to use, like variable wait-times, or programmatic back-off if you observe the query time / MB to be decreasing.

I won't start with over-engineering this. I'll start with a light sip, but how to break up the queries? 

### Date ranges

This is a good one, and I'll use this going forward to get our periodic updates. For this first big pull, we need to 


### Logical units

Bundling sites together by organizational ID, hyrdologic unit, or by region may be a better way of doing things if we're trying to fill in the whole backlog of available data. For daily updates, querying for all records for the last few days by state (or multi-state region, as you can add a list of states) would be a clean way of minimizing your queries while keeping the data size down. This is something we'll need to test out and decide experimentally. 

Lets take a look at the site data again. 

In [ ]:
%load_ext autoreload
%autoreload 2

from hydroflow.ingest_wqp import ingest_wqp_site_data_by_state, ingest_wqp_site_results

state_name = "Washington"
site_type = "Stream"
raw_df, metadata = ingest_wqp_site_data_by_state(state_name, site_type)
print(raw_df.shape)

site_id = "USGS-12422000"
results_df, results_metadata = ingest_wqp_site_results(site_id)
print(results_df.shape)

# Basic assertions to make sure we got data back
assert raw_df.shape[0] > 0, "Expected to retrieve at least one site for Washington streams."
assert results_df.shape[0] > 0, f"Expected to retrieve at least one result for site {site_id}."

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Fetching Stream monitoring sites in Washington...
Found 17936 Stream monitoring sites in state:US:53, type:Stream for the period None to None.
(17936, 37)
(414, 63)
